# Sceptical base-rate benchmark

Runs conditions from `data/base_rate/benchmark.csv`: 10 vignettes × 6 variants (open / MC numeric / MC full × with / without stated probabilities). Tune **`DEBUG_MAX_PROMPTS`** and **`MAX_OUTPUT_TOKENS`** in the setup cell (after `FORCE_REPO_REFRESH`).

**Kaggle setup:** Add-ons → Secrets → `GITHUB_TOKEN` (GitHub PAT with `repo` scope, toggled ON). Settings → Internet ON. **Run the setup cell below first.**

The notebook downloads `FrancisGanong-N/sceptical_llms` from the **`master`** branch (not `main`). If setup prints `Repo has max_prompts support: False`, set `FORCE_REPO_REFRESH = True` in the setup cell and re-run it before the dry-run cell.

**Publishing:** Run the dry-run cell before `%choose`.

**Quota:** If you see a `403` about `max_output_tokens` / estimated cost, re-run setup with `FORCE_REPO_REFRESH = True` (the benchmark caps output at 512 tokens and disables reasoning). For a full run you may still need more Kaggle Model Proxy quota, or use a small `DEBUG_MAX_PROMPTS` while debugging.

**Outputs:** The dry-run cell writes `base_rate_merged_results.csv` and `base_rate_score_pivot.csv` under `/kaggle/working/`. To download them, **Save Version** (Commit) the notebook, open that version’s **Output** tab, and download the CSVs. Copy into `data/base_rate/` locally, then open `benchmark/base-rate-results.ipynb`.

After the run, a pivoted summary is printed: **rows = model**, **columns = condition** (`open_probs`, `open_no_probs`, `mc_numeric_probs`, …). Each cell is the **mean normative score** (1 = normative answer, 0 = otherwise) across vignettes in that condition.

In [ ]:
import csv
import io
import json
import shutil
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

GITHUB_OWNER = "FrancisGanong-N"
GITHUB_REPO = "sceptical_llms"
GITHUB_BRANCH = "master"  # default branch on github.com/FrancisGanong-N/sceptical_llms
KAGGLE_REPO_DIR = Path("/kaggle/working") / GITHUB_REPO
EXPECTED_BENCHMARK_PROMPTS = 54
FORCE_REPO_REFRESH = True  # set False only after setup prints quota-safe repo OK
# Set to None for the full 60-prompt benchmark; 2 is enough to debug CSV/pivot output.
DEBUG_MAX_PROMPTS = 18
MAX_OUTPUT_TOKENS = 128


def benchmark_prompt_count(root: Path) -> int:
    benchmark_csv = root / "data" / "base_rate" / "benchmark.csv"
    if not benchmark_csv.is_file():
        return 0
    with benchmark_csv.open(newline="", encoding="utf-8") as handle:
        return sum(1 for _ in csv.DictReader(handle))


def has_fresh_repo(root: Path) -> bool:
    base_rate_py = root / "benchmarks" / "base_rate.py"
    tasks_py = root / "benchmarks" / "base_rate_tasks.py"
    if not tasks_py.is_file():
        return False
    tasks_source = tasks_py.read_text(encoding="utf-8")
    return (
        base_rate_py.is_file()
        and benchmark_prompt_count(root) >= EXPECTED_BENCHMARK_PROMPTS
        and "CONDITION_COLUMN_ORDER" in base_rate_py.read_text(encoding="utf-8")
        and "max_prompts" in tasks_source
        and "_prompt_llm" in tasks_source
        and "_llm_extra_api_params" in tasks_source
        and "response = llm.prompt(prompt)" not in tasks_source
    )


def repo_has_quota_safe_prompting(root: Path) -> bool:
    tasks_py = root / "benchmarks" / "base_rate_tasks.py"
    if not tasks_py.is_file():
        return False
    source = tasks_py.read_text(encoding="utf-8")
    return "_prompt_llm" in source and "_llm_extra_api_params" in source and "response = llm.prompt(prompt)" not in source


def download_repo_from_github() -> Path:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = (
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
        f"/zipball/{GITHUB_BRANCH}"
    )
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "kaggle-sceptical-llms-benchmark",
        },
    )

    staging = Path("/kaggle/working") / "_repo_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(staging)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub download failed ({exc.code}) for "
            f"github.com/{GITHUB_OWNER}/{GITHUB_REPO}@{GITHUB_BRANCH}: {body[:300]}"
        ) from exc

    extracted = next(p for p in staging.iterdir() if p.is_dir())
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    shutil.copytree(extracted, KAGGLE_REPO_DIR)
    shutil.rmtree(staging)
    return KAGGLE_REPO_DIR


def bootstrap_repo() -> Path:
    if not FORCE_REPO_REFRESH and has_fresh_repo(KAGGLE_REPO_DIR):
        return KAGGLE_REPO_DIR

    for candidate in (Path.cwd(), Path.cwd().parent):
        if not FORCE_REPO_REFRESH and has_fresh_repo(candidate):
            return candidate

    if not Path("/kaggle/working").is_dir():
        raise RuntimeError(
            "Could not find a fresh sceptical-llms repo (need "
            f"{EXPECTED_BENCHMARK_PROMPTS} rows in benchmark.csv). Run from the repo, "
            "or on Kaggle set GITHUB_TOKEN and enable Internet, then re-run this cell."
        )

    root = download_repo_from_github()
    if not repo_has_quota_safe_prompting(root):
        raise RuntimeError(
            "GitHub download succeeded but benchmarks/base_rate_tasks.py is still "
            "missing quota-safe prompting (_prompt_llm). Push latest master to "
            f"github.com/{GITHUB_OWNER}/{GITHUB_REPO}, then re-run this cell."
        )
    return root


ROOT = bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Re-run setup after a repo refresh must not keep stale benchmark modules.
for module_name in list(sys.modules):
    if module_name == "benchmarks" or module_name.startswith("benchmarks."):
        del sys.modules[module_name]

print("Repo root:", ROOT)
print("Benchmark rows:", benchmark_prompt_count(ROOT))
print("Repo has max_prompts support:", has_fresh_repo(ROOT))
print("Repo has quota-safe prompting:", repo_has_quota_safe_prompting(ROOT))
print("DEBUG_MAX_PROMPTS:", DEBUG_MAX_PROMPTS)
print("MAX_OUTPUT_TOKENS:", MAX_OUTPUT_TOKENS)
if not repo_has_quota_safe_prompting(ROOT):
    raise RuntimeError(
        "Stale benchmarks/base_rate_tasks.py (still calls llm.prompt directly). "
        "Set FORCE_REPO_REFRESH = True and re-run this setup cell."
    )

In [ ]:
import csv
from pathlib import Path

import kaggle_benchmarks as kbench
from benchmarks.base_rate import print_score_pivots
from benchmarks import base_rate_tasks
from benchmarks.base_rate_tasks import evaluate_base_rate_benchmark

tasks_source = (Path(ROOT) / "benchmarks" / "base_rate_tasks.py").read_text(encoding="utf-8")
if "response = llm.prompt(prompt)" in tasks_source:
    raise RuntimeError(
        "Stale base_rate_tasks.py on disk. Re-run the setup cell "
        "(FORCE_REPO_REFRESH = True) before this cell."
    )

CANDIDATE_LLMS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.0-flash",
    "anthropic/claude-haiku-4-5@20251001",
    "qwen/qwen3-next-80b-a3b-instruct",
]


def _brief_error(exc: BaseException) -> str:
    message = str(exc).strip().splitlines()[0] if str(exc).strip() else type(exc).__name__
    if len(message) > 200:
        message = message[:197] + "..."
    return f"{type(exc).__name__}: {message}"


available = set(kbench.llms.keys())
llm_errors: list[str] = []
selected_llm_name = None
runs = score = merged_path = pivot_path = pivot = None

for llm_name in CANDIDATE_LLMS:
    if llm_name not in available:
        summary = f"{llm_name}: KeyError (not in kbench.llms)"
        llm_errors.append(summary)
        print(summary)
        continue
    print(f"Trying {llm_name}...")
    try:
        runs, score, merged_path, pivot_path, pivot = evaluate_base_rate_benchmark(
            kbench.llms[llm_name],
            max_prompts=DEBUG_MAX_PROMPTS,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            n_jobs=1,
        )
        selected_llm_name = llm_name
        print(f"Using {llm_name}")
        break
    except Exception as exc:
        summary = f"{llm_name}: {_brief_error(exc)}"
        llm_errors.append(summary)
        print(summary)

if selected_llm_name is None:
    raise RuntimeError(
        "All candidate LLMs failed:\n" + "\n".join(llm_errors)
    )

print("Normative accuracy:", f"{score.normative_accuracy:.1%}")
print("Bias index:", f"{score.bias_index:.1%}")
print("Parse rate:", f"{score.parse_rate:.1%}")
print("Merged results:", merged_path)
print("Score pivot CSV:", pivot_path)

# Explicit notebook display (stdout from %choose is easy to miss on Kaggle).
with merged_path.open(encoding="utf-8") as handle:
    print_score_pivots(list(csv.DictReader(handle)))

try:
    from IPython.display import display

    display(pivot)
except ImportError:
    pass

## Download results for local analysis

On Kaggle, `/kaggle/working/` files are only kept if you **Save Version** (top-right **Save Version** → **Save & Run All** or **Quick Save** after a successful dry-run).

1. Open the saved version → **Output** tab → download the two CSVs.
2. Locally, place them in `data/base_rate/` (same names).
3. Run `benchmark/base-rate-results.ipynb`.

In [ ]:
import pandas as pd

for path in (merged_path, pivot_path):
    print(path, f"({path.stat().st_size:,} bytes)")

merged_df = pd.read_csv(merged_path)
print(f"\nMerged rows: {len(merged_df)}  |  model: {selected_llm_name}")
merged_df.head()

In [ ]:
%choose base_rate_normative_accuracy